# IFRS 9 and CECL ECL

**Purpose:** Walk through a compact expected-credit-loss workflow using the `finstack.statements_analytics` bindings.

**Prerequisites:** Familiarity with PD, LGD, EAD, and the difference between 12-month and lifetime ECL.

**In this notebook:** We classify a loan across Stage 1, Stage 2, and Stage 3 conditions, compute 12-month and lifetime ECL, and then add probability-weighted macro scenarios.


## Concept

Under IFRS 9, Stage 1 exposures use **12-month ECL**, while Stage 2 and Stage 3 exposures use **lifetime ECL**. In practice, the workflow is usually:

1. Define the exposure state.
2. Classify the stage from delinquency or credit deterioration.
3. Build a cumulative PD term structure.
4. Compute stage-appropriate ECL.
5. Weight multiple macro scenarios into a final allowance.


In [ ]:
from finstack.statements_analytics import (
    Exposure,
    classify_stage,
    compute_ecl,
    compute_ecl_weighted,
)


def banner(title: str) -> None:
    print(f"\n{'=' * 72}")
    print(title)
    print('=' * 72)


base_exposure = Exposure(
    id="CORP-LOAN-001",
    ead=1_000_000.0,
    lgd=0.45,
    eir=0.06,
    remaining_maturity=5.0,
    current_pd=0.032,
    origination_pd=0.030,
    dpd=0,
)

sicr_exposure = Exposure(
    id="CORP-LOAN-001-SICR",
    ead=1_000_000.0,
    lgd=0.45,
    eir=0.06,
    remaining_maturity=5.0,
    current_pd=0.045,
    origination_pd=0.030,
    dpd=0,
)

default_exposure = Exposure(
    id="CORP-LOAN-001-NPL",
    ead=1_000_000.0,
    lgd=0.45,
    eir=0.06,
    remaining_maturity=5.0,
    current_pd=0.25,
    origination_pd=0.030,
    dpd=120,
)

base_pd_schedule = [
    (1.0, 0.008),
    (2.0, 0.015),
    (3.0, 0.022),
    (4.0, 0.028),
    (5.0, 0.032),
]

sicr_pd_schedule = [
    (1.0, 0.015),
    (2.0, 0.025),
    (3.0, 0.033),
    (4.0, 0.040),
    (5.0, 0.045),
]

upside_pd_schedule = [
    (1.0, 0.004),
    (2.0, 0.008),
    (3.0, 0.012),
    (4.0, 0.016),
    (5.0, 0.020),
]

downside_pd_schedule = [
    (1.0, 0.025),
    (2.0, 0.045),
    (3.0, 0.065),
    (4.0, 0.085),
    (5.0, 0.100),
]

scenarios = [
    (0.60, base_pd_schedule),
    (0.20, upside_pd_schedule),
    (0.20, downside_pd_schedule),
]

base_exposure

## Stage classification and allowance build

The next cell mirrors the flat example script, but in notebook form so you can inspect intermediate outputs. It shows the stage trigger for each exposure, compares Stage 1 versus Stage 2 allowance size, and then applies IFRS 9-style scenario weights.


In [ ]:
banner("Synthetic exposure")
print(base_exposure)

banner("Stage classification")
for label, exposure in (
    ("Performing", base_exposure),
    ("SICR", sicr_exposure),
    ("90+ DPD", default_exposure),
):
    stage, reason = classify_stage(
        exposure,
        pd_delta_stage2=0.01,
        dpd_30_trigger=True,
        dpd_90_trigger=True,
    )
    print(f"{label:<12} -> {stage:<8} | trigger: {reason}")

ecl_12m = compute_ecl(
    ead=base_exposure.ead,
    pd_schedule=base_pd_schedule,
    lgd=base_exposure.lgd,
    eir=base_exposure.eir,
    max_horizon_years=base_exposure.remaining_maturity,
    bucket_width_years=0.25,
    stage="stage1",
)

ecl_lifetime = compute_ecl(
    ead=sicr_exposure.ead,
    pd_schedule=sicr_pd_schedule,
    lgd=sicr_exposure.lgd,
    eir=sicr_exposure.eir,
    max_horizon_years=sicr_exposure.remaining_maturity,
    bucket_width_years=0.25,
    stage="stage2",
)

banner("ECL computation")
print(f"Stage 1 (12-month) ECL       : ${ecl_12m:,.2f}")
print(f"Stage 2 (lifetime) ECL       : ${ecl_lifetime:,.2f}")
print(f"Lifetime / 12m multiple      : {ecl_lifetime / ecl_12m:.1f}x")

weighted_12m = compute_ecl_weighted(
    ead=base_exposure.ead,
    scenarios=scenarios,
    lgd=base_exposure.lgd,
    eir=base_exposure.eir,
    max_horizon=base_exposure.remaining_maturity,
    stage="stage1",
)

weighted_lifetime = compute_ecl_weighted(
    ead=base_exposure.ead,
    scenarios=scenarios,
    lgd=base_exposure.lgd,
    eir=base_exposure.eir,
    max_horizon=base_exposure.remaining_maturity,
    stage="stage2",
)

banner("Probability-weighted macro scenarios")
print(f"Weighted Stage 1 ECL         : ${weighted_12m:,.2f}")
print(f"Weighted Stage 2 ECL         : ${weighted_lifetime:,.2f}")
print("\nPer-scenario Stage 1 ECL:")
for label, (weight, schedule) in zip(("base", "upside", "downside"), scenarios):
    scenario_ecl = compute_ecl(
        ead=base_exposure.ead,
        pd_schedule=schedule,
        lgd=base_exposure.lgd,
        eir=base_exposure.eir,
        max_horizon_years=base_exposure.remaining_maturity,
        bucket_width_years=0.25,
        stage="stage1",
    )
    print(f"  {label:<8} (w={weight:.0%}) -> ${scenario_ecl:,.2f}")


## Takeaways

- `classify_stage()` makes the stage trigger explicit, which is useful for audit trails.
- `compute_ecl()` separates **term structure inputs** from **stage selection**.
- `compute_ecl_weighted()` is the clean bridge from single-scenario expected loss to IFRS 9 probability weighting.
- The main modeling judgment still lives outside the API: how you build the PD scenarios and when you call SICR.


In [ ]:
{
    "stage1_ecl": round(ecl_12m, 2),
    "stage2_ecl": round(ecl_lifetime, 2),
    "weighted_stage1_ecl": round(weighted_12m, 2),
    "weighted_stage2_ecl": round(weighted_lifetime, 2),
}
